In [ ]:
# Purpose - compare spectra from multiple files

findices = [170] #[130] #[4]

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
#from BinnedSpectrum import BinnedSpectrum

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

runs = [TristanRun(findex) for findex in findices]
boilerplate = "TODO" #run.boilerplate + f"tstep = {run.steps[0]}"

In [ ]:
def plot_spectrum(subplot, gammas, run):
    log_bins = 10 ** np.linspace(-4, 4, 100) #300
    hist, bins = np.histogram((gammas - 1) / run.sigma, bins = log_bins, density = True)

    subplot.set_xlabel("$(\gamma - 1) / \sigma$")
    subplot.set_ylabel("$dN / d\gamma$")
    subplot.set_title(f"Electron Spectra")
    subplot.grid(visible = True)
    subplot.axvline(1, color = 'black', linestyle = '--', label = '$\gamma - 1 = \sigma$', alpha = 0.5)
    subplot.axvline(1 / run.sigma, color = 'orange', linestyle = '--', label = '$\gamma - 1 = 1$', alpha = 0.5)

    outflow_gamma = (1 - run.beta_A ** 2) ** -0.5
    subplot.axvline((outflow_gamma - 1) / run.sigma, color = 'purple', linestyle = '--', label = '$\gamma = \gamma_A$', alpha = 0.5)

    subplot.legend()
    
    subplot.loglog(bins[1:], hist)

In [ ]:
interval = [0.02, 0.6]
interval = [0.6, 10]
interval = [0.02, 10]

xcoeff = 1 # x axis is xcoeff * (gamma - 1)
tot_normalize = True

In [ ]:
# Plot Domain and Spectrum

species = 1

plt.close()
fig, ax1 = plt.subplots(nrows = 1, ncols = 1, figsize = (6, 4))

colors = plt.cm.viridis(np.linspace(0, 1, len(runs)))
colors = ['red', 'blue']

tot_normalize = True

for i, run in enumerate(runs):
    label = True

    for sel_ts in [28, 29]: # [42, 50]
        #bs = run.find_total_spectrum(sel_ts, species)
        bs = r

        yvals = bs.calc_hist()

        if (not tot_normalize):
            yvals *= bs.total

        if (label):
            ax1.loglog(bs.ebins * xcoeff, yvals, linewidth = 1,
                    markersize = 1.5, label = f'ylen = {run.my0 / run.c_omp}, ts = {sel_ts}, tot = {bs.total:.2e}')
            #label = False
        else:
            ax1.loglog(bs.ebins * xcoeff, bs.calc_hist() * bs.total, linewidth = 1,
                        markersize = 1.5, color=colors[i])

plt.suptitle("")

line_values = { #in gamma 
    'sigma': 1 + run.sigma,
    '1': 2,
    #'gamma_A': (1 - run.beta_A ** 2) ** -0.5,
    'gamma_spec': (1 + run.beta_A ** 2) / (1 - run.beta_A ** 2)
}

line_colors = { #in gamma 
    'sigma': 'black',
    '1': 'black',
    'gamma_A': 'cyan',
    'gamma_spec': 'purple'
}

for line in line_values:

    ax1.axvline((line_values[line] - 1) * xcoeff, color = line_colors[line] , linestyle = '--', alpha = 0.5, label = line)

title = "Energy Spectra"

#for i in range(len(runs)):
    #title += f"\n {runs[i].code}, $\gamma_{{mean}} - 1 = {means[i]:.2f}, p = {probs[i]:.2e}$"

ax1.set_title(title)
ax1.grid(visible = True)
ax1.set_xlim([1e-4, 1e2])

#ax1.axvspan(interval[0], interval[1], color='yellow', alpha=0.3, label='Interval')

#ax2.set_ylim([1, 1e12])
ax1.legend()
#ax2.loglog(spec_bins, spec_counts, linewidth = 1, markersize = 1.5)

plt.xlabel("$\gamma - 1$")  

if (tot_normalize):
    plt.ylabel("$f(\gamma)$")
else:
    plt.ylabel("$dN / d\gamma$")

plt.show()

In [ ]:
# Define the left and right bounds
left_bound = 4  # You can change this value as needed
right_bound = 100      # You can change this value as needed

# Find the spectrum at ts = 42
bs_42 = run.find_total_spectrum(42, species)

hist = bs_42.calc_hist()

# Calculate the total energy
total_energy = np.sum(hist * bs_42.ebins)

# Calculate the energy within the bounds
energy_within_bounds = np.sum(hist[(bs_42.ebins > left_bound) & (bs_42.ebins < right_bound)] * bs_42.ebins[(bs_42.ebins > left_bound) & (bs_42.ebins < right_bound)])

# Calculate the fraction
fraction_within_bounds = energy_within_bounds / total_energy

print(f"Fraction of total energy within bounds {left_bound} and {right_bound}: {fraction_within_bounds}")

In [ ]:
print(run.spec_nx)
print(run.spec_ny)

In [ ]:
print(0 / 0)

In [ ]:
# Close any existing plots
plt.close()

# Create a new figure and axis
fig, ax2 = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))

# Iterate over each run
for i, run in enumerate(runs):
    cutoffs = []
    timesteps = run.steps

    # Calculate the energy cutoff for each timestep
    for sel_ts in timesteps:
        bs = run.find_total_spectrum(sel_ts, species)

        cutoff = bs.calc_cutoff()
        cutoffs.append(cutoff)

    # Plot the energy cutoff vs. timestep
    ax2.plot(timesteps, cutoffs, label=f'Run {run.code}', color=colors[i])

# Set axis labels and title
ax2.set_xlabel('$t \Omega$')
ax2.set_ylabel('Energy Cutoff ($\gamma - 1$)')
ax2.set_title('Energy Cutoff vs. Timestep')
ax2.grid(visible=True)
ax2.legend()

# Show the plot
plt.show()